# Chapter 3: seven ways a chart lies, then the redraw

Each lie is drawn before and after on Kestrel data. Lies 2, 4 and 5 use the chapter's own worked numbers,
because the dataset holds no sales against temperature series and no single week of filler oxygen. Synthetic data only.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

DATA = Path("../data")   # run this notebook from the notebooks folder
def load(name, **kw):
    return pd.read_csv(DATA / f"{name}.csv", **kw)

def pair(title):
    fig, (a, b) = plt.subplots(1, 2, figsize=(10, 3.4))
    fig.suptitle(title)
    a.set_title("Before", loc="left"); b.set_title("After", loc="left")
    return fig, a, b

## 1. The truncated axis: cellar loss in February and March

In [ ]:
st = load("Stage")
cel = st[(st.Stage == "Cellar") & st.Month.isin(["2026-02-01", "2026-03-01"])]
pct = (100 * cel.LossHL / cel.InHL).round(1).tolist()
print(pct, "gap", round(pct[1] / pct[0], 2))
fig, a, b = pair("Cellar loss, percent of cast")
for ax, low in ((a, 2.0), (b, 0)):
    ax.bar(["Feb", "Mar"], pct, color="0.5"); ax.set_ylim(low, 2.6); ax.set_ylabel("Percent")
plt.show()

## 2. Two axes that fake a link (the chapter's toy series, seed 32)

In [ ]:
rng = np.random.default_rng(32)
days = np.arange(60)
sales = 270 + np.cumsum(rng.normal(0, 4, 60))
temp = 18 + np.cumsum(rng.normal(0, 0.05, 60))
fig, a, b = pair("Tank temperature and sales")
a.plot(days, sales, color="tab:blue"); a2 = a.twinx(); a2.plot(days, temp, color="tab:orange")
a2.set_ylim(temp.min(), temp.max())
b.plot(days, sales, color="tab:blue"); b.set_ylabel("Sales (hL a day)")
b2 = b.inset_axes([0, -0.75, 1, 0.55]); b2.plot(days, temp, color="tab:orange"); b2.set_ylabel("Temperature (degrees C)")
plt.show()

## 3. The pie with nine slices: stop reasons on the can line

In [ ]:
stops = load("Stop")
top = stops.groupby("StopReason").Minutes.sum().sort_values(ascending=False)
print(top.size, "reasons used;", "top nine shown")
nine = top.head(9)
fig, a, b = pair("Stop minutes by reason, four weeks")
a.pie(nine, labels=None)
shown = pd.concat([top.head(8), pd.Series({"All other reasons": top.iloc[8:].sum()})])[::-1]
b.barh(shown.index, shown.values, color=["0.7"] * (len(shown) - 3) + ["tab:blue"] * 3); b.set_xlabel("Minutes")
plt.show()

## 4. Smoothing that hides the spike (the chapter's numbers: six days at 20 ppb, one at 140)

In [ ]:
do = pd.Series([20, 20, 20, 20, 20, 20, 140], index=range(1, 8))
print("seven day mean", round(do.mean(), 1), "ppb")
fig, a, b = pair("Dissolved oxygen at the filler feed, ppb")
a.plot([7], [do.mean()], "o-"); a.set_ylim(0, 150)
b.plot(do.index, do.values, "o", color="0.6"); b.axhline(do.mean(), color="tab:blue"); b.set_ylim(0, 150)
plt.show()

## 5. The average of averages (Monday 2 samples at 2.0, Tuesday 8 at 3.0)

In [ ]:
rows = pd.DataFrame({"day": ["Mon"] * 2 + ["Tue"] * 8, "plato": [2.0] * 2 + [3.0] * 8})
mean_of_means = rows.groupby("day").plato.mean().mean()
print("mean of means", mean_of_means, "| mean of rows", rows.plato.mean())

## 6. Gaps drawn as slopes: the six hour dropout on F11

In [ ]:
rd = load("Readings", parse_dates=["Time"])
f11 = rd[(rd.Tag == "Temp") & (rd.Tank == "F11")].set_index("Time").Value
full = f11.reindex(pd.date_range(f11.index.min(), f11.index.max(), freq="min"))
print(full.isna().sum() / 60, "hours missing")
fig, a, b = pair("F11 temperature, degrees C")
a.plot(f11.index, f11.values); b.plot(full.index, full.values)
b.axvspan(full[full.isna()].index.min(), full[full.isna()].index.max(), color="0.9")
fig.autofmt_xdate(); plt.show()

## 7. Mixed units on one scale: hL lost and percent lost

In [ ]:
vol = load("Volume")
lost = vol.CastHL - vol.DispatchedHL
fig, a, b = pair("Loss by month, 2026")
a.plot(lost.values, label="hL"); a.plot((100 * lost / vol.CastHL).values, label="percent"); a.legend()
b.plot(lost.values); b.set_ylabel("Loss (hL)")
b2 = b.inset_axes([0, -0.75, 1, 0.55]); b2.plot((100 * lost / vol.CastHL).values, color="tab:orange"); b2.set_ylabel("Loss (percent)")
plt.show()

**What to look at.** Run the ten questions of section 3.3 on each before chart. Each one fails one question.